In [2]:
import pandas as pd #importing dataset
attributes = ["buying", "maint", "doorNum", "persons", "sizeBoot", "safety", "classValue"] #creating a list/name for each attribute
CarData = pd.read_csv("car.data", names=attributes) #reads the data from the data file into a data frame (table)
CarData["carID"]=["Car "+str(i) for i in range(1,len(CarData)+1)] #creates a unique ID for each car

In [3]:
print ("Enter price of the car as low, med, high or vhigh: ") #price prompt for the user input
buying = input("buying: ") #variable to store user input
print ("User has input: ", buying)

Enter price of the car as low, med, high or vhigh: 
User has input:  high


In [4]:
row = {"buying":buying} #telling it what to look for
#doors, persons and lug_boot isn't relevant to the problem I want to solve

facts = {f"{attribute}={value}" for attribute, value in row.items()} #makes attribute & value pairs into facts

rules = [
    ({"buying=low"}, ("class_value=acc")), #cheap car? what a bargain, why not?
    ({"buying=vhigh"}, ("class_value=unacc")) #too expensive, don't want it
]

In [5]:
def forward_chain(facts, rules): #def(ining) a function & allowing it to use facts & rules within it. for running the user input against the rules
    for conditions, conclusion in rules: #for loop, keeping it simple. conditions = must match the rule before a rule can be used. conclusion = result of meeting rule requirements.
        if conditions.issubset(facts): #python method that returns true if everything in 'facts' is met, if not, returns false
            facts.add(conclusion) #adds the conclusion (outcome) of matching rules to current facts
            return conclusion #returns with the conclusion
    return "class_value=unacc" #default return if nothing matches

In [6]:
result = forward_chain(facts, rules) #result is what returns from the forward_chain function
print ("This car is rated as ", result) #prints result

This car is rated as  class_value=unacc


In [7]:
import random 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix #importing the tools needed for testing/accuracy from scikit

random.seed(38) #makes random shuffle create the same result every run
shuffled = CarData[attributes].values.tolist() #puts data from dataset into new list. [attributes] makes it only use things listed in the 'attributes'.
random.shuffle(shuffled) #shuffles data in said list
cut = int(0.7 * len(shuffled)) #70% training data and 30% testing data
train, test = shuffled[:cut], shuffled[cut:] #splits the data into train or test

print("Training records:", len(train)) #prints how many cars were used for training
print("Testing records:", len(test)) #prints how many cars were used for testing

actual = [] #stores all of the actual classifications of each car
predictions = [] #stores what the model predicts the classification of each car is

for row in test: #for loop that checks every test car
    facts = { #creates facts from the data
        "buying=" + row[0], #gets the price and joins it together with 'buying=' to create a fact
    }
    result = forward_chain(facts, rules) #calls the function, model starts making predictions
    actual.append(row[6]) #brings the actual classifications into the 'actual' row from earlier, for comparison
    predictions.append(result.replace("class_value=", "")) #stores the prediction, using replace to remove the "class_value" cuz it isn't needed

#accuracy stuff
accuracy = accuracy_score(actual, predictions) #using scikit, compares the two lists and calculates how many were correct (so i dont have to do maths)

print("\nTEST RESULTS") #\n for new line, for neatness
print("Correct Predictions Made:", sum(a == p for a, p in zip(actual, predictions))) #zip pairs the actual & prediction, a==p checks if actual & prediction match, sum counts the correct ones
print("Total Predictions Made:", len(actual)) #counts how many cars are in 'actual'
print("Accuracy:", round(accuracy * 100, 2), "%") #shows the accuracy as a percentage

Training records: 1209
Testing records: 519

TEST RESULTS
Correct Predictions Made: 309
Total Predictions Made: 519
Accuracy: 59.54 %
